# Практикум по теме "Магические методы (часть 3): менеджеры контекста, `__new__`, `__del__"`

## 🎯 Цель практикума
Закрепить на практике использование:
- Контекстных менеджеров (`__enter__` / `__exit__`)
- Настоящего конструктора `__new__`
- Финализатора `__del__` (и понять, почему его лучше избегать)

## 📋 Структура
1. **Базовые задачи** (3 шт.) – для отработки основ
2. **Задачи повышенной сложности** (2 шт.) – требуют творческого подхода
3. **Домашнее задание** (3 шт.) – для самостоятельной работы

> 💡 **Совет:** перед выполнением задач освежите в памяти материал лекции. Не стесняйтесь использовать подсказки в коде.

## 🟢 Базовые задачи

### Задача 1. Контекстный менеджер `Logger`
**Условие:** Реализуйте контекстный менеджер `Logger`, который записывает в лог-файл время входа в блок `with` и время выхода, а также информацию о возникших исключениях (если были).

**Требования:**
- При входе в блок записывается строка: `[ВХОД] YYYY-MM-DD HH:MM:SS`
- При выходе (без исключения): `[ВЫХОД] YYYY-MM-DD HH:MM:SS`
- Если произошло исключение, записывается: `[ИСКЛЮЧЕНИЕ] тип: сообщение`
- Имя файла передаётся в конструктор.

```python
# Пример использования:
with Logger('log.txt'):
    print("Что-то делаем...")
    # Здесь может быть исключение

In [ ]:
# Шаблон для решения
import datetime

class Logger:
    def __init__(self, filename):
        # сохраняем имя файла
        pass

    def __enter__(self):
        # открыть файл, записать время входа
        pass

    def __exit__(self, exc_type, exc_val, exc_tb):
        # записать время выхода или информацию об исключении
        # не забыть закрыть файл
        pass

# Проверка
with Logger('test_log.txt'):
    print("Работа без ошибок")
    # raise ValueError("Ой!")  # раскомментируйте для проверки исключения

### Задача 2. Лимитированный класс через `__new__`
**Условие:** Создайте класс `LimitedInstances`, который позволяет создать не более `max_instances` экземпляров. При попытке создать лишний экземпляр должно возникать исключение `RuntimeError` с сообщением "Превышен лимит создания экземпляров".

**Требования:**
- Лимит задаётся атрибутом класса `max_instances`
- Счётчик текущих экземпляров должен храниться в классе
- Используйте `__new__` для контроля создания

```python
# Пример использования:
class MyClass(LimitedInstances):
    max_instances = 2

a = MyClass()
b = MyClass()
c = MyClass()  # Должно быть исключение

In [ ]:
# Шаблон для решения
class LimitedInstances:
    _count = 0

    def __new__(cls, *args, **kwargs):
        # проверяем лимит
        pass

# Реализуйте дочерний класс
class MyClass(LimitedInstances):
    max_instances = 2

# Проверка
try:
    a = MyClass()
    b = MyClass()
    c = MyClass()
except RuntimeError as e:
    print(f"Ошибка: {e}")

### Задача 3. Наблюдение за удалением (`__del__`)
**Условие:** Напишите класс `Watcher`, который при создании выводит сообщение `"Создан объект {id}"`, а при удалении (вызове `__del__`) выводит `"Удалён объект {id}"`. Создайте несколько объектов, удалите ссылки на них вручную и с помощью сборщика мусора.

**Требования:**
- Используйте `__init__` и `__del__`
- Продемонстрируйте, когда вызывается `__del__`
- Создайте циклическую ссылку и посмотрите, как это влияет на удаление (можно использовать `gc`)

```python
# Пример использования:
w1 = Watcher()
w2 = Watcher()
del w1  # должно вывести сообщение об удалении
# создайте циклическую ссылку: w3.ref = w4; w4.ref = w3

In [ ]:
# Шаблон для решения
import gc

class Watcher:
    def __init__(self):
        # выводим сообщение о создании
        pass

    def __del__(self):
        # выводим сообщение об удалении
        pass

# Демонстрация
print("--- Простое удаление ---")
w1 = Watcher()
w2 = Watcher()
del w1

print("--- Циклическая ссылка ---")
w3 = Watcher()
w4 = Watcher()
w3.ref = w4
w4.ref = w3
del w3
del w4
# Принудительный сбор мусора
gc.collect()

## 🔴 Задачи повышенной сложности

### Задача 4. Контекстный менеджер для временной смены кодировки вывода
**Условие:** Реализуйте контекстный менеджер `encoding`, который временно меняет кодировку стандартного вывода (`sys.stdout`) на указанную. При выходе восстанавливает исходную.

**Требования:**
- Конструктор принимает имя кодировки (например, `'cp1251'`)
- При входе сохраняется текущая кодировка `sys.stdout.encoding` (если есть) и меняется на новую (можно присвоить `sys.stdout.encoding` напрямую, но это не рекомендуется; лучше использовать `sys.stdout.reconfigure(encoding=...)`, если Python 3.7+)
- При выходе восстанавливается исходная кодировка
- Обработайте случай, если `sys.stdout` не поддерживает изменение кодировки

```python
# Пример использования:
with encoding('utf-16'):
    print("Привет, мир!")  # этот текст будет в кодировке utf-16
print("А это уже в исходной")

In [ ]:
# Шаблон для решения
import sys

class encoding:
    def __init__(self, enc):
        self.new_enc = enc
        self.old_enc = None

    def __enter__(self):
        # сохранить старую кодировку и установить новую
        pass

    def __exit__(self, *args):
        # восстановить старую кодировку
        pass

# Проверка
with encoding('utf-16'):
    sys.stdout.write("Тест\n")

### Задача 5. Класс "Одноразовый" с помощью `__new__`
**Условие:** Реализуйте класс `OneTime`, который можно создать только один раз. При повторном вызове конструктора должен возвращаться тот же самый объект (как Singleton), но при этом если к объекту уже обращались, то должно выводиться предупреждение `"Объект уже был создан, возвращается существующий"`.

**Требования:**
- Используйте `__new__` для контроля создания
- Добавьте атрибут `created`, который хранит, был ли уже создан объект
- В `__init__` добавьте флаг, чтобы не инициализировать объект повторно (как в лекции)

```python
# Пример использования:
a = OneTime(10)
b = OneTime(20)  # должно предупредить и вернуть тот же объект, значение не изменится
print(a is b)    # True
print(a.value)   # 10

In [ ]:
# Шаблон для решения
class OneTime:
    _instance = None
    _created = False

    def __new__(cls, *args, **kwargs):
        # реализуйте логику
        pass

    def __init__(self, value):
        # инициализируйте только один раз
        pass

# Проверка
a = OneTime(42)
b = OneTime(100)
print(a is b)
print(a.value)  # должно быть 42

## 🏠 Домашнее задание

### Задача 6. Контекстный менеджер для временной директории
**Условие:** Реализуйте контекстный менеджер `temp_dir`, который создаёт временную директорию (например, с помощью `tempfile.mkdtemp`), переходит в неё, а при выходе удаляет её вместе со всем содержимым и возвращается в исходную директорию.

**Требования:**
- Используйте `os.chdir` для смены директории
- Используйте `shutil.rmtree` для удаления временной папки
- Гарантируйте возврат в исходную директорию даже при исключении

```python
# Пример использования:
with temp_dir() as tmp:
    # здесь мы находимся во временной папке
    with open('test.txt', 'w') as f:
        f.write('hello')
    # после выхода папка удалится

### Задача 7. Пул объектов через `__new__`
**Условие:** Создайте класс `Pool`, который управляет пулом объектов `Worker`. Пул имеет максимальный размер. При запросе нового `Worker` через `Pool.get()` должен возвращаться либо свободный существующий объект, либо создаваться новый (если лимит не превышен). После использования объект возвращается в пул методом `Pool.release(worker)`.

**Требования:**
- Используйте `__new__` для создания объектов `Worker` (можно просто обычный класс)
- В классе `Pool` храните список свободных объектов и общее количество созданных
- Если свободных нет и лимит достигнут, `get()` должен ждать (можно просто бросить исключение или вернуть `None` – на ваше усмотрение)

```python
# Пример использования:
pool = Pool(max_size=2)
w1 = pool.get()
w2 = pool.get()
w3 = pool.get()  # должно вернуть None или вызвать исключение
pool.release(w2)
w3 = pool.get()  # теперь получится



### Задача 8. Проблема циклических ссылок и `__del__`
**Условие:** Напишите два класса: `Parent` и `Child`, которые ссылаются друг на друга. У каждого определён `__del__`, выводящий сообщение. Создайте экземпляры, удалите ссылки на них и проанализируйте, вызвался ли `__del__`. Затем используйте модуль `gc` для принудительной сборки и объясните результат.

**Требования:**
- В `__init__` сохраняйте ссылки друг на друга
- Удалите внешние ссылки (`del parent, del child`)
- Вызовите `gc.collect()` и посмотрите, появятся ли сообщения от `__del__`
- Добавьте комментарии в коде с объяснением

```python
# Пример:
class Parent:
    def __init__(self, child):
        self.child = child
    def __del__(self):
        print("Parent deleted")

class Child:
    def __init__(self, parent):
        self.parent = parent
    def __del__(self):
        print("Child deleted")

# Создание циклической ссылки
p = Parent(None)
c = Child(p)
p.child = c
# удаляем ссылки
del p
del c
# ... что произойдёт?